# 06 · Simüle Veri ile Eğitim ve Değerlendirme

`05_simulate_dataset.ipynb` ile üretilen haritalarda: kayıt bazlı train/val/test bölmesi, tek kanallı CNN eğitimi
(model seçimi yalnızca val), test bölümünde bağımsız değerlendirme ve **tespit başarımının SNR'a göre değişimi**.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "passive_radar").is_dir())
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
import json
from dataclasses import asdict, replace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from passive_radar.config import CLASS_NAMES, ProjectConfig
from passive_radar.data.dataset import RangeDopplerDataset, make_data_loader
from passive_radar.data.indexing import (
    SPLIT_NAMES, assert_no_group_leakage, records_in_split, save_split_csv, split_by_source,
)
from passive_radar.evaluation.metrics import collect_predictions, metrics_from_predictions
from passive_radar.evaluation.plots import plot_confusion_matrix, plot_rd_maps, plot_training_history
from passive_radar.inference.predictor import predict_image
from passive_radar.models.checkpoint import load_checkpoint
from passive_radar.models.cnn import RangeDopplerCNN, count_trainable_parameters
from passive_radar.simulation.dataset_builder import load_simulated_records
from passive_radar.training.runtime import resolve_device, set_seed
from passive_radar.training.trainer import Trainer

config = ProjectConfig.default(PROJECT_ROOT)
paths = config.paths
out_dir = paths.simulated_outputs_dir
rd_config = config.rd_map
model_config = replace(config.model, in_channels=rd_config.num_channels, pooled_size=(2, 2), global_pool="max")
training_config = replace(config.training, batch_size=32, learning_rate=1e-3, max_epochs=30, early_stopping_patience=5)
for name, section in (("rd_map", rd_config), ("model", model_config), ("training", training_config)):
    print(f"{name}: {asdict(section)}")

## 1. Kayıt bazlı bölme

In [ ]:
records = load_simulated_records(paths.simulated_dir)
split_config = replace(config.split, max_holdout_group_fraction=0.2)  # simulated recordings are similar in size
assigned = split_by_source(records, split_config)
assert_no_group_leakage(assigned)
save_split_csv(assigned, out_dir / "splits.csv", relative_to=paths.project_root)
pd.DataFrame([
    {"split": s, "maps": len(rs), "target": sum(r.label == 1 for r in rs), "recordings": len({r.source_id for r in rs})}
    for s in SPLIT_NAMES for rs in [records_in_split(assigned, s)]
])

## 2. Eğitim (early stopping, en iyi model val kaybına göre)

In [ ]:
loaders = {
    split: make_data_loader(
        RangeDopplerDataset(records_in_split(assigned, split), rd_config, cache_images=True),
        training_config.batch_size, shuffle=(split == "train"), seed=training_config.seed)
    for split in SPLIT_NAMES
}
set_seed(training_config.seed)
device = resolve_device(training_config.device)
model = RangeDopplerCNN(model_config)
print("device:", device, "| parameters:", f"{count_trainable_parameters(model):,}")

trainer = Trainer(model, training_config, device, paths.simulated_checkpoint_path, CLASS_NAMES, rd_config)
history = trainer.fit(loaders["train"], loaders["val"])
history.save_json(out_dir / "training_history.json")
plot_training_history(history, save_path=out_dir / "figures" / "training_history.png")
plt.show()

## 3. Test bölümünde bağımsız değerlendirme

In [ ]:
checkpoint = load_checkpoint(paths.simulated_checkpoint_path, device=device)
results = {}
for split in ("val", "test"):
    predictions = collect_predictions(checkpoint.model, loaders[split], device)
    results[split] = (predictions, metrics_from_predictions(predictions))
pd.DataFrame({s: {k: v for k, v in m.items() if k != "confusion_matrix"} for s, (_, m) in results.items()}).T

In [ ]:
test_predictions, test_metrics = results["test"]
(out_dir / "test_metrics.json").write_text(json.dumps({s: m for s, (_, m) in results.items()}, indent=2))
plot_confusion_matrix(test_metrics["confusion_matrix"], CLASS_NAMES, save_path=out_dir / "figures" / "confusion_matrix_test.png")
plt.show()

## 4. Tespit olasılığı ve yanlış alarm oranı — SNR'a göre

In [ ]:
metadata = pd.read_csv(paths.simulated_dir / "metadata.csv")
metadata["path"] = [str(paths.simulated_dir / p) for p in metadata["path"]]
test_records = records_in_split(assigned, "test")
frame = pd.DataFrame({"path": [str(r.path) for r in test_records],
                      "true": test_predictions.y_true, "pred": test_predictions.y_pred,
                      "p_target": test_predictions.positive_probability}).merge(metadata, on="path")

false_alarm_rate = (frame.loc[frame["true"] == 0, "pred"] == 1).mean()
targets = frame[frame["true"] == 1].copy()
targets["snr_bin"] = pd.cut(targets["target_snr_db"], bins=np.arange(-45, -24, 2.5), include_lowest=True)
pd_by_snr = targets.groupby("snr_bin", observed=True)["pred"].agg(detection_probability="mean", maps="size")
print(f"false alarm rate (no_target predicted as target): {false_alarm_rate:.3f}")

axis = pd_by_snr["detection_probability"].plot(marker="o", figsize=(8, 3.5), ylim=(0, 1.05), grid=True,
                                                title="Test: detection probability vs. input SNR")
axis.set_xlabel("per-sample echo SNR [dB] (processing gain ~ +47 dB)")
plt.tight_layout()
plt.savefig(out_dir / "figures" / "detection_vs_snr.png", dpi=120)
plt.show()
pd_by_snr

## 5. Hatalı sınıflandırılan örnekler ve tek harita tahmini

In [ ]:
wrong = frame[frame["true"] != frame["pred"]].head(3)
if wrong.empty:
    print("No misclassified test maps.")
else:
    plot_rd_maps([np.load(p) for p in wrong["path"]],
                 [f"true={CLASS_NAMES[t]} p_target={q:.2f} SNR={s}" for t, q, s in
                  wrong[["true", "p_target", "target_snr_db"]].itertuples(index=False)],
                 config.simulation.max_range_km, config.simulation.max_doppler_hz,
                 markers=[(r, d) if t == 1 else None for t, r, d in
                          wrong[["true", "target_range_km", "target_doppler_hz"]].itertuples(index=False)])
    plt.show()

print(predict_image(test_records[0].path, checkpoint_path=paths.simulated_checkpoint_path))

## 6. Yorum ve sınırlılıklar

* Etiketler fiziksel olarak tanımlıdır (yankı eklendi / eklenmedi); başarım SNR'a bağlıdır. Düşük SNR'da tespitin
  zorlaşması beklenen ve radar açısından anlamlı bir sonuçtur.
* Senaryo idealdir: gürültüsüz referans, doğrudan yol girişimi / clutter yok, tek hedef. Gerçekçiliği artırmak için
  doğrudan yol + clutter eklenip pyAPRiL'in ECA clutter bastırması uygulanabilir.
* Gerçek ölçüm verisine genelleme doğrulanmamıştır.